# Aplicação Estratégica dos Resultados

Este notebook transforma as previsões do modelo final em indicadores estratégicos para apoiar a análise da alfabetização no Brasil.

Os objetivos desta etapa são:

- identificar municípios com maior risco educacional;
- analisar padrões territoriais e regionais;
- relacionar o risco previsto aos principais fatores identificados pelo modelo;
- avaliar a distância em relação às metas de alfabetização;
- produzir indicadores de priorização para políticas públicas;
- gerar visualizações para a entrega final.

As análises utilizam o modelo final previamente selecionado, sem novos ajustes de hiperparâmetros ou threshold.

# 1. Configuração

Nesta etapa são importadas as bibliotecas utilizadas para reconstrução das previsões, agregações territoriais e visualizações estratégicas.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

import numpy as np
from itertools import combinations

from pathlib import Path

import matplotlib.pyplot as plt

Configuração do Base de Dados para buscar o nome dos municípios direto na fonte

In [0]:
dbutils.widgets.text(
    "billing_project_id",
    "fiap-tech-challenge2",
    "Billing Project GCP"
)

billing_project_id = dbutils.widgets.get(
    "billing_project_id"
)

credencial_gcp = dbutils.secrets.get(
    scope="fiap-tech-challenge",
    key="gcp-service-account-b64"
)

Diretórios para salvar os gráficos

In [0]:
cwd = Path.cwd()

repo_root = next(
    (pasta for pasta in [cwd, *cwd.parents] if (pasta / "README.md").exists()), cwd
)

images_dir = repo_root / "images"
tables_dir = repo_root / "reports" / "tables"

images_dir.mkdir(parents=True, exist_ok=True)

tables_dir.mkdir(parents=True, exist_ok=True)

print("Raiz do repositório:", repo_root)
print("Pasta de imagens:", images_dir)
print("Pasta de tabelas:", tables_dir)

# 2. Carregamento dos Produtos do Modelo

As previsões finais de 2024 e as importâncias das variáveis foram produzidas e persistidas no notebook de avaliação final.

Dessa forma, este notebook utiliza diretamente os produtos analíticos já gerados, sem necessidade de retreinar ou reaplicar o modelo.

Essa separação mantém as responsabilidades dos notebooks bem definidas:

- o notebook 10 é responsável pela avaliação e geração das previsões finais;
- o notebook 11 é responsável pela transformação dessas previsões em inteligência estratégica.

In [0]:
df_predicoes_2024 = spark.table("workspace.gold.ml_predicoes_2024")

df_importancia = spark.table("workspace.gold.ml_feature_importance")

Validação

In [0]:
display(
    df_predicoes_2024.agg(
        F.count("*").alias("qtd_alunos"),
        F.countDistinct("id_municipio").alias("qtd_municipios"),
        F.round(F.avg("probabilidade_0") * 100, 2).alias("risco_medio_pct"),
    )
)

In [0]:
display(
    df_importancia
    .orderBy(F.desc("importancia"))
)

# 3. Indicadores de Risco por Município

As previsões individuais são agregadas para o nível municipal.

O risco educacional é representado pela probabilidade média prevista de um aluno pertencer à classe de não alfabetizados.

Também são calculadas as taxas prevista e observada de alfabetização para apoiar a interpretação dos resultados.

## 3.1 Município + Rede

In [0]:
df_risco_rede = df_predicoes_2024.groupBy("ano", "id_municipio", "rede").agg(
    F.count("*").alias("qtd_alunos"),
    F.round(F.avg("label") * 100, 2).alias("taxa_real_alfabetizacao_pct"),
    # Estimativa probabilística do modelo
    F.round(F.avg("probabilidade_1") * 100, 2).alias("score_alfabetizacao_pct"),
    # Resultado da classificação após threshold 0.55
    F.round(F.avg("prediction") * 100, 2).alias("pct_classificado_alfabetizado"),
    F.round(F.avg(1 - F.col("prediction")) * 100, 2).alias(
        "pct_classificado_nao_alfabetizado"
    ),
    F.round(F.avg("probabilidade_0") * 100, 2).alias("risco_medio_pct"),
    F.avg("inse_2021").alias("inse_medio"),
    F.avg("ideb_saeb_portugues_2021").alias("ideb_portugues_medio"),
    F.avg("ideb_indicador_rendimento_2021").alias("rendimento_medio"),
    F.avg("censo_pct_internet_aprendizagem").alias("internet_aprendizagem_media"),
    F.avg("censo_pct_biblioteca_sala_leitura").alias("biblioteca_media"),
    F.avg("censo_docentes_por_100_alunos").alias("docentes_100_alunos_medio"),
    F.avg("log_pib_per_capita").alias("log_pib_per_capita_medio"),
)

## 3.2 Município

In [0]:
df_risco_municipio = df_predicoes_2024.groupBy("ano", "id_municipio").agg(
    F.count("*").alias("qtd_alunos"),
    F.round(F.avg("label") * 100, 2).alias("taxa_real_alfabetizacao_pct"),
    # Probabilidade média estimada de alfabetização
    F.round(F.avg("probabilidade_1") * 100, 2).alias("score_alfabetizacao_pct"),
    # Percentual classificado como alfabetizado após threshold 0.55
    F.round(F.avg("prediction") * 100, 2).alias("pct_classificado_alfabetizado"),
    F.round(F.avg(1 - F.col("prediction")) * 100, 2).alias(
        "pct_classificado_nao_alfabetizado"
    ),
    F.round(F.avg("probabilidade_0") * 100, 2).alias("risco_medio_pct"),
    F.avg("inse_2021").alias("inse_medio"),
    F.avg("ideb_saeb_portugues_2021").alias("ideb_portugues_medio"),
    F.avg("ideb_indicador_rendimento_2021").alias("rendimento_medio"),
    F.avg("censo_pct_internet_aprendizagem").alias("internet_aprendizagem_media"),
    F.avg("censo_pct_biblioteca_sala_leitura").alias("biblioteca_media"),
    F.avg("censo_docentes_por_100_alunos").alias("docentes_100_alunos_medio"),
    F.avg("log_pib_per_capita").alias("log_pib_per_capita_medio"),
)

## 3.3 Erro agregado

In [0]:
df_risco_municipio = df_risco_municipio.withColumn(
    "erro_taxa_pp",
    F.round(
        F.abs(
            F.col("taxa_real_alfabetizacao_pct")
            - F.col("score_alfabetizacao_pct")
        ),
        2,
    ),
)

# 4. Enriquecimento Territorial

Os indicadores municipais são enriquecidos com nome do município, Unidade da Federação e região.

Essas informações permitem transformar as previsões em análises territoriais mais interpretáveis para gestores públicos.

In [0]:
df_diretorio_municipio = (
    spark.read.format("bigquery")
    .option("credentials", credencial_gcp)
    .option("parentProject", billing_project_id)
    .load("basedosdados.br_bd_diretorios_brasil.municipio")
)

df_municipio_dim = df_diretorio_municipio.select(
    "id_municipio", F.col("nome").alias("municipio"), "sigla_uf"
).dropDuplicates(["id_municipio"])

Criando as regiões

In [0]:
regioes = [
    ("AC", "Norte"),
    ("AP", "Norte"),
    ("AM", "Norte"),
    ("PA", "Norte"),
    ("RO", "Norte"),
    ("RR", "Norte"),
    ("TO", "Norte"),
    ("AL", "Nordeste"),
    ("BA", "Nordeste"),
    ("CE", "Nordeste"),
    ("MA", "Nordeste"),
    ("PB", "Nordeste"),
    ("PE", "Nordeste"),
    ("PI", "Nordeste"),
    ("RN", "Nordeste"),
    ("SE", "Nordeste"),
    ("DF", "Centro-Oeste"),
    ("GO", "Centro-Oeste"),
    ("MT", "Centro-Oeste"),
    ("MS", "Centro-Oeste"),
    ("ES", "Sudeste"),
    ("MG", "Sudeste"),
    ("RJ", "Sudeste"),
    ("SP", "Sudeste"),
    ("PR", "Sul"),
    ("RS", "Sul"),
    ("SC", "Sul"),
]

df_regioes = spark.createDataFrame(regioes, ["sigla_uf", "regiao"])

In [0]:
df_risco_municipio = df_risco_municipio.join(
    df_municipio_dim, on="id_municipio", how="left"
).join(df_regioes, on="sigla_uf", how="left")

Agregando nas previsões

In [0]:
df_predicoes_geo = df_predicoes_2024.join(
    df_municipio_dim, on="id_municipio", how="left"
).join(df_regioes, on="sigla_uf", how="left")

# 5. Faixas de Risco Municipal

Os municípios são classificados em quatro faixas relativas de risco com base na probabilidade média prevista de não alfabetização.

As faixas representam posições relativas dentro da população analisada e não limites absolutos de desempenho educacional.

In [0]:
window_risco = Window.orderBy(F.col("risco_medio_pct"))

df_risco_municipio = df_risco_municipio.withColumn(
    "quartil_risco", F.ntile(4).over(window_risco)
).withColumn(
    "faixa_risco",
    F.when(F.col("quartil_risco") == 4, "Muito alto")
    .when(F.col("quartil_risco") == 3, "Alto")
    .when(F.col("quartil_risco") == 2, "Moderado")
    .otherwise("Baixo"),
)

Distribuição

In [0]:
display(
    df_risco_municipio.groupBy("faixa_risco")
    .agg(
        F.count("*").alias("qtd_municipios"),
        F.round(F.avg("risco_medio_pct"), 2).alias("risco_medio_pct"),
    )
    .orderBy(F.desc("risco_medio_pct"))
)

## 5.1 Municípios com Maior Risco

O ranking abaixo identifica os municípios em que o modelo atribuiu maior probabilidade média de não alfabetização.

A priorização é baseada nas previsões do modelo e deve ser utilizada como instrumento de apoio à decisão, não como diagnóstico causal.

In [0]:
display(
    df_risco_municipio.select(
        "municipio",
        "sigla_uf",
        "regiao",
        "qtd_alunos",
        "risco_medio_pct",
        "score_alfabetizacao_pct",
        "taxa_real_alfabetizacao_pct",
        "inse_medio",
        "ideb_portugues_medio",
        "rendimento_medio",
        "faixa_risco",
    )
    .orderBy(F.desc("risco_medio_pct"))
    .limit(30)
)

# 6. Relação entre Risco e Principais Fatores

Os municípios são comparados pelas faixas de risco para verificar como os principais fatores identificados pela Random Forest variam entre os grupos.

A análise é descritiva e não representa relação causal.

In [0]:
display(
    df_risco_municipio.groupBy("faixa_risco")
    .agg(
        F.count("*").alias("qtd_municipios"),
        F.round(F.avg("risco_medio_pct"), 2).alias("risco_medio_pct"),
        F.round(F.avg("inse_medio"), 3).alias("inse_medio"),
        F.round(F.avg("ideb_portugues_medio"), 2).alias("ideb_portugues_medio"),
        F.round(F.avg("rendimento_medio"), 3).alias("rendimento_medio"),
        F.round(F.avg("internet_aprendizagem_media") * 100, 2).alias(
            "internet_aprendizagem_pct"
        ),
        F.round(F.avg("biblioteca_media") * 100, 2).alias("biblioteca_pct"),
        F.round(F.avg("docentes_100_alunos_medio"), 2).alias("docentes_100_alunos"),
    )
    .orderBy(F.desc("risco_medio_pct"))
)

# 7. Análise Regional

As previsões são agregadas por região para identificar diferenças territoriais no risco educacional.

Também são analisados os principais fatores associados ao modelo em cada região.

In [0]:
df_risco_regiao = df_predicoes_geo.groupBy("regiao").agg(
    F.count("*").alias("qtd_alunos"),
    F.countDistinct("id_municipio").alias("qtd_municipios"),
    F.round(F.avg("label") * 100, 2).alias("taxa_real_alfabetizacao_pct"),
    # Probabilidade média estimada de alfabetização
    F.round(F.avg("probabilidade_1") * 100, 2).alias("score_alfabetizacao_pct"),
    F.round(F.avg("prediction") * 100, 2).alias("pct_classificado_alfabetizado"),
    F.round(F.avg("probabilidade_0") * 100, 2).alias("risco_medio_pct"),
    F.round(F.avg("inse_2021"), 3).alias("inse_medio"),
    F.round(F.avg("ideb_saeb_portugues_2021"), 2).alias("ideb_portugues_medio"),
    F.round(F.avg("ideb_indicador_rendimento_2021"), 3).alias("rendimento_medio"),
    F.round(F.avg("log_pib_per_capita"), 3).alias("log_pib_per_capita_medio"),
)

In [0]:
display(df_risco_regiao.orderBy(F.desc("risco_medio_pct")))

## 7.1 Análise por UF

In [0]:
df_risco_uf = df_predicoes_geo.groupBy("regiao", "sigla_uf").agg(
    F.count("*").alias("qtd_alunos"),
    F.countDistinct("id_municipio").alias("qtd_municipios"),
    F.round(F.avg("probabilidade_0") * 100, 2).alias("risco_medio_pct"),
    # Probabilidade média estimada de alfabetização
    F.round(F.avg("probabilidade_1") * 100, 2).alias("score_alfabetizacao_pct"),
    F.round(F.avg("prediction") * 100, 2).alias("pct_classificado_alfabetizado"),
    F.round(F.avg("label") * 100, 2).alias("taxa_real_alfabetizacao_pct"),
    F.round(F.avg("inse_2021"), 3).alias("inse_medio"),
    F.round(F.avg("ideb_saeb_portugues_2021"), 2).alias("ideb_portugues_medio"),
    F.round(F.avg("ideb_indicador_rendimento_2021"), 3).alias("rendimento_medio"),
)

In [0]:
display(df_risco_uf.orderBy(F.desc("risco_medio_pct")))

# 8. Regiões com Padrões Semelhantes

Para explorar similaridades territoriais, são comparados os perfis das regiões considerando risco previsto, INSE, desempenho histórico em Língua Portuguesa, rendimento escolar e contexto econômico.

As variáveis são padronizadas antes do cálculo da distância entre os perfis.

In [0]:
pdf_regioes = df_risco_regiao.select(
    "regiao",
    "risco_medio_pct",
    "inse_medio",
    "ideb_portugues_medio",
    "rendimento_medio",
    "log_pib_per_capita_medio",
).toPandas()

In [0]:
metricas_perfil = [
    "risco_medio_pct",
    "inse_medio",
    "ideb_portugues_medio",
    "rendimento_medio",
    "log_pib_per_capita_medio",
]

for coluna in metricas_perfil:
    media = pdf_regioes[coluna].mean()
    desvio = pdf_regioes[coluna].std(ddof=0)

    pdf_regioes[f"z_{coluna}"] = (pdf_regioes[coluna] - media) / desvio

In [0]:
colunas_padronizadas = [f"z_{coluna}" for coluna in metricas_perfil]

pares_regioes = []

for i, j in combinations(pdf_regioes.index, 2):
    regiao_a = pdf_regioes.loc[i, "regiao"]

    regiao_b = pdf_regioes.loc[j, "regiao"]

    distancia = np.sqrt(
        (
            (
                pdf_regioes.loc[i, colunas_padronizadas]
                - pdf_regioes.loc[j, colunas_padronizadas]
            )
            ** 2
        ).sum()
    )

    pares_regioes.append((regiao_a, regiao_b, float(distancia)))

In [0]:
df_similaridade_regioes = spark.createDataFrame(
    pares_regioes, ["regiao_a", "regiao_b", "distancia"]
)

display(df_similaridade_regioes.orderBy("distancia"))

# 9. Relação com Metas Municipais

As previsões são relacionadas às metas municipais de alfabetização.

Para evitar o problema identificado na fase anterior, a comparação utiliza a taxa prevista pelo modelo e a meta de alfabetização, em vez de comparar a meta com a própria taxa de referência presente na tabela de metas.

Para metas posteriores a 2024, a análise representa um indicador de distância atual até a meta e não uma previsão direta do resultado futuro.

## 9.1 Carregamento

In [0]:
df_metas = spark.table("workspace.silver.metas")

df_metas.printSchema()

## 9.2 Preparação  das Metas Municipais

A tabela de metas utiliza `id_geografia` para representar diferentes níveis territoriais.

In [0]:
# Seleciona registros municipais e padroniza as chaves
df_metas_municipais_base = (
    df_metas.filter(F.length(F.col("id_geografia").cast("string")) == 7)
    .filter(F.col("meta_alfabetizacao").isNotNull())
    .filter(F.col("rede").isin("Estadual", "Municipal"))
    .withColumn("id_municipio", F.col("id_geografia").cast("string"))
    .withColumn(
        "rede_codigo",
        F.when(F.col("rede") == "Estadual", F.lit("2")).when(
            F.col("rede") == "Municipal", F.lit("3")
        ),
    )
)

In [0]:
window_revisao_meta = Window.partitionBy(
    "id_municipio", "rede_codigo", "ano_meta"
).orderBy(F.desc("ano_referencia"))

df_metas_municipais = (
    df_metas_municipais_base.withColumn(
        "rn_revisao", F.row_number().over(window_revisao_meta)
    )
    .filter(F.col("rn_revisao") == 1)
    .drop("rn_revisao")
)

In [0]:
df_metas_municipais = df_metas_municipais.withColumn(
    "meta_alfabetizacao_pct",
    F.when(
        F.col("meta_alfabetizacao") <= 1, F.col("meta_alfabetizacao") * 100
    ).otherwise(F.col("meta_alfabetizacao")),
)

## 9.3 Próxima Meta Após 2024

In [0]:
window_proxima_meta = Window.partitionBy("id_municipio", "rede_codigo").orderBy(
    "ano_meta"
)

df_proxima_meta = (
    df_metas_municipais.filter(F.col("ano_meta") > 2024)
    .withColumn("rn_meta", F.row_number().over(window_proxima_meta))
    .filter(F.col("rn_meta") == 1)
    .select(
        "id_municipio",
        F.col("rede_codigo").alias("rede"),
        "ano_meta",
        "meta_alfabetizacao_pct",
    )
)

## 9.4 Distância até a Meta

In [0]:
df_risco_meta = (
    df_risco_rede.withColumn("rede", F.col("rede").cast("string"))
    .join(df_proxima_meta, on=["id_municipio", "rede"], how="inner")
    .withColumn(
        "distancia_score_meta_pp",
        F.round(
            F.col("score_alfabetizacao_pct") - F.col("meta_alfabetizacao_pct"),
            2,
        ),
    )
)

In [0]:
df_risco_meta = df_risco_meta.join(
    df_municipio_dim, on="id_municipio", how="left"
).join(df_regioes, on="sigla_uf", how="left")

In [0]:
display(
    df_risco_meta.select(
        "municipio",
        "sigla_uf",
        "rede",
        "ano_meta",
        "qtd_alunos",
        "score_alfabetizacao_pct",
        "meta_alfabetizacao_pct",
        "distancia_score_meta_pp",
        "risco_medio_pct",
    )
    .orderBy("distancia_score_meta_pp")
    .limit(30)
)

# 10. Priorização Estratégica

A priorização estratégica combina duas dimensões complementares:

- **risco educacional**, representado pela probabilidade média de não alfabetização estimada pelo modelo;
- **distância em relação à meta**, representada pela diferença entre o score médio de alfabetização e a próxima meta disponível.

A classificação é relativa aos municípios e redes analisados. Portanto, as categorias de prioridade não representam limites oficiais de política pública, mas um instrumento analítico para ordenar contextos que podem demandar maior atenção.

O número estimado de alunos em situação de risco é utilizado como indicador adicional de escala, permitindo diferenciar localidades com alta vulnerabilidade relativa de localidades em que o problema potencialmente alcança um número maior de estudantes.

## 10.1 Construção dos indicadores de priorização

In [0]:
# Quanto maior, mais distante o contexto está da meta
df_priorizacao = (
    df_risco_meta
    .withColumn("defasagem_meta_pp", F.greatest(-F.col("distancia_score_meta_pp"), F.lit(0.0)))
    .withColumn("indice_exposicao_risco", F.round(F.col("qtd_alunos") * F.col("risco_medio_pct") / 100, 0).cast("long"),)
    .withColumn(
        "rede_descricao",
        F.when(F.col("rede") == "2", "Estadual")
        .when(F.col("rede") == "3", "Municipal")
        .otherwise("Outras"),
    )
)

## 10.2 Quartis de risco e defasagem

In [0]:
# Contextos que ainda apresentam distância em relação à meta
df_priorizacao_com_defasagem = df_priorizacao.filter(F.col("defasagem_meta_pp") > 0)

# Ranking relativo apenas entre contextos abaixo da meta
window_defasagem_prioridade = Window.orderBy(F.col("defasagem_meta_pp"))

df_priorizacao_com_defasagem = df_priorizacao_com_defasagem.withColumn(
    "percentil_defasagem", F.percent_rank().over(window_defasagem_prioridade)
).withColumn(
    "prioridade",
    F.when(F.col("percentil_defasagem") >= (2 / 3), "Muito alta")
    .when(F.col("percentil_defasagem") >= (1 / 3), "Alta")
    .otherwise("Moderada"),
)

# Contextos cujo score já alcança ou supera a meta
df_priorizacao_sem_defasagem = (
    df_priorizacao.filter(F.col("defasagem_meta_pp") == 0)
    .withColumn("percentil_defasagem", F.lit(0.0))
    .withColumn("prioridade", F.lit("Acompanhamento"))
)

# Reúne novamente todos os contextos
df_priorizacao = (
    df_priorizacao_com_defasagem.unionByName(df_priorizacao_sem_defasagem)
    .withColumn(
        "ordem_prioridade",
        F.when(F.col("prioridade") == "Muito alta", 4)
        .when(F.col("prioridade") == "Alta", 3)
        .when(F.col("prioridade") == "Moderada", 2)
        .otherwise(1),
    )
    .withColumn(
        "rede_descricao",
        F.when(F.col("rede") == "2", "Estadual")
        .when(F.col("rede") == "3", "Municipal")
        .otherwise("Outras"),
    )
)

## 10.3 Distribuição das Prioridades

A distribuição abaixo mostra quantos contextos município-rede foram classificados em cada nível relativo de prioridade.

Também são apresentados o risco médio, a defasagem média em relação à meta e o número estimado de alunos associados ao risco.

In [0]:
display(
    df_priorizacao.groupBy("prioridade")
    .agg(
        F.count("*").alias("qtd_contextos"),
        F.round(F.avg("risco_medio_pct"), 2).alias("risco_medio_pct"),
        F.round(F.avg("defasagem_meta_pp"), 2).alias("defasagem_media_meta_pp"),
        F.sum("indice_exposicao_risco").alias("indice_exposicao_risco"),
    )
    .orderBy(F.desc("defasagem_media_meta_pp"))
)

## 10.4 Contextos de Maior Prioridade

O ranking a seguir apresenta os contextos classificados com maior prioridade relativa.

Dentro de um mesmo nível de prioridade, contextos com maior número estimado de alunos associados ao risco são apresentados primeiro, incorporando uma dimensão de escala à análise sem alterar a classificação de vulnerabilidade.

In [0]:
display(
    df_priorizacao.select(
        "municipio",
        "sigla_uf",
        "regiao",
        "rede_descricao",
        "qtd_alunos",
        "risco_medio_pct",
        "score_alfabetizacao_pct",
        "ano_meta",
        "meta_alfabetizacao_pct",
        "distancia_score_meta_pp",
        "defasagem_meta_pp",
        "indice_exposicao_risco",
        "prioridade",
    )
    .orderBy(
        F.desc("ordem_prioridade"),
        F.desc("defasagem_meta_pp"),
        F.desc("indice_exposicao_risco"),
    )
    .limit(30)
)

## 10.5 Prioridade por escala do problema

In [0]:
display(
    df_priorizacao.select(
        "municipio",
        "sigla_uf",
        "regiao",
        "rede_descricao",
        "qtd_alunos",
        "risco_medio_pct",
        "score_alfabetizacao_pct",
        "defasagem_meta_pp",
        "indice_exposicao_risco",
        "prioridade",
    )
    .orderBy(F.desc("indice_exposicao_risco"))
    .limit(30)
)

## 10.6 Resumo por Região

In [0]:
display(
    df_priorizacao.groupBy("regiao", "prioridade")
    .agg(
        F.count("*").alias("qtd_contextos"),
        F.sum("qtd_alunos").alias("qtd_alunos"),
        F.round(F.avg("risco_medio_pct"), 2).alias("risco_medio_pct"),
        F.round(F.avg("defasagem_meta_pp"), 2).alias("defasagem_media_meta_pp"),
        F.sum("indice_exposicao_risco").alias("indice_exposicao_risco"),
    )
    .orderBy("regiao", F.desc("defasagem_media_meta_pp"))
)

## 10.7 Interpretação da Priorização Estratégica

A priorização foi construída a partir da **defasagem em relação à meta**, isto é, da diferença entre o score médio de alfabetização estimado pelo modelo e a meta definida para o município e rede.

Por exemplo, se o score de alfabetização for de 60% e a meta for de 75%, a defasagem será de **15 pontos percentuais**. Quanto maior essa defasagem, maior a prioridade relativa do contexto.

Os contextos classificados como **Muito alta prioridade** apresentaram defasagem média de **17,04 pontos percentuais**, enquanto os de prioridade **Alta** apresentaram média de **8,01 pontos percentuais**.

Além da defasagem, também foi analisado o **índice de exposição ao risco**, que combina o tamanho da população estudantil com o risco médio estimado. Esse indicador ajuda a diferenciar municípios pequenos, mas muito vulneráveis, de grandes municípios em que o risco pode atingir um volume maior de alunos.

Assim, a análise permite duas leituras complementares:

- **vulnerabilidade**: contextos mais distantes de suas metas;
- **escala**: contextos em que o risco está associado a um número maior de alunos.

O Nordeste concentrou o maior número de contextos classificados como Muito alta prioridade. Ao mesmo tempo, grandes municípios de outras regiões também apresentaram elevada exposição ao risco.

Esses indicadores devem ser utilizados como apoio à priorização e ao planejamento de políticas públicas, e não como critérios automáticos de decisão.

# 11. Persistência dos Produtos Estratégicos

Os principais produtos analíticos construídos nesta etapa são persistidos na camada Gold.

A persistência permite reutilizar os resultados sem recalcular as análises, além de garantir rastreabilidade e disponibilizar tabelas prontas para consumo por relatórios, dashboards e outras aplicações analíticas.

## 11.1 Risco por município

In [0]:
(
    df_risco_municipio.write.format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.gold.ml_risco_municipio_2024")
)

## 11.2 Risco por UF

In [0]:
(
    df_risco_uf.write.format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.gold.ml_risco_uf_2024")
)

## 11.3 Risco por região

In [0]:
(
    df_risco_regiao.write.format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.gold.ml_risco_regiao_2024")
)

## 11.4 Priorização educacional

In [0]:
(
    df_priorizacao.write.format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.gold.ml_priorizacao_educacional_2024")
)

Validação

In [0]:
tabelas_estrategicas = [
    "workspace.gold.ml_risco_municipio_2024",
    "workspace.gold.ml_risco_uf_2024",
    "workspace.gold.ml_risco_regiao_2024",
    "workspace.gold.ml_priorizacao_educacional_2024",
]

resultado_validacao = []

for tabela in tabelas_estrategicas:
    resultado_validacao.append((tabela, spark.table(tabela).count()))

df_validacao_gold = spark.createDataFrame(
    resultado_validacao, ["tabela", "qtd_registros"]
)

display(df_validacao_gold)

# 12. Visualizações Estratégicas

As principais conclusões da análise são consolidadas em visualizações voltadas à comunicação executiva.

Os gráficos desta seção destacam os fatores mais relevantes para o modelo, diferenças territoriais de risco, contextos com maior exposição e a distribuição das prioridades estratégicas.

In [0]:
def salvar_figura(fig, nome):
    caminho = images_dir / f"{nome}.png"

    fig.savefig(
        caminho,
        dpi=200,
        bbox_inches="tight"
    )

    print(f"PNG salvo: {caminho}")

## 12.1 Importância das Features

In [0]:
pdf_importancia = (
    df_importancia
    .orderBy(F.desc("importancia"))
    .limit(10)
    .toPandas()
    .sort_values("importancia_pct")
)

fig, ax = plt.subplots(figsize=(11, 6))

ax.barh(
    pdf_importancia["feature"],
    pdf_importancia["importancia_pct"]
)

ax.set_title("Principais variáveis do modelo")
ax.set_xlabel("Importância (%)")
ax.set_ylabel("Variável")

plt.tight_layout()

salvar_figura(
    fig,
    "feature_importance"
)

plt.show()
plt.close(fig)

## 12.2 Risco médio por região

In [0]:
pdf_regiao = (
    df_risco_regiao.select("regiao", "risco_medio_pct")
    .orderBy(F.desc("risco_medio_pct"))
    .toPandas()
)

fig, ax = plt.subplots(figsize=(9, 5))

ax.bar(pdf_regiao["regiao"], pdf_regiao["risco_medio_pct"])

ax.set_title("Risco médio estimado por região")
ax.set_xlabel("Região")
ax.set_ylabel("Risco médio (%)")

plt.tight_layout()

salvar_figura(fig, "risco_por_regiao")

plt.show()
plt.close(fig)

## 12.3 Municípios com maior risco
Para facilitar a leitura executiva, a visualização considera municípios com pelo menos 100 alunos na população analisada. O filtro é utilizado apenas no gráfico e não altera os indicadores ou rankings persistidos na camada Gold.

In [0]:
pdf_top_risco = (
    df_risco_municipio.filter(F.col("qtd_alunos") >= 100)
    .orderBy(F.desc("risco_medio_pct"))
    .select("municipio", "sigla_uf", "risco_medio_pct")
    .limit(15)
    .toPandas()
)

pdf_top_risco["municipio_uf"] = (
    pdf_top_risco["municipio"] + " - " + pdf_top_risco["sigla_uf"]
)

pdf_top_risco = pdf_top_risco.sort_values("risco_medio_pct")

fig, ax = plt.subplots(figsize=(11, 7))

ax.barh(pdf_top_risco["municipio_uf"], pdf_top_risco["risco_medio_pct"])

ax.set_title("Municípios com maior risco educacional estimado")
ax.set_xlabel("Risco médio (%)")
ax.set_ylabel("Município")

plt.tight_layout()

salvar_figura(fig, "municipios_maior_risco")

plt.show()
plt.close(fig)

## 12.4 Maior exposição ao risco

In [0]:
pdf_exposicao = (
    df_priorizacao.orderBy(F.desc("indice_exposicao_risco"))
    .select("municipio", "sigla_uf", "indice_exposicao_risco")
    .limit(15)
    .toPandas()
)

pdf_exposicao["municipio_uf"] = (
    pdf_exposicao["municipio"] + " - " + pdf_exposicao["sigla_uf"]
)

pdf_exposicao = pdf_exposicao.sort_values("indice_exposicao_risco")

fig, ax = plt.subplots(figsize=(11, 7))

ax.barh(pdf_exposicao["municipio_uf"], pdf_exposicao["indice_exposicao_risco"])

ax.set_title("Contextos com maior exposição ao risco educacional")
ax.set_xlabel("Índice de exposição ao risco")
ax.set_ylabel("Município")

plt.tight_layout()

salvar_figura(fig, "exposicao_risco")

plt.show()
plt.close(fig)

## 12.5 Distribuição das prioridades

In [0]:
ordem_prioridade = ["Acompanhamento", "Moderada", "Alta", "Muito alta"]

pdf_prioridade = (
    df_priorizacao.groupBy("prioridade")
    .agg(F.count("*").alias("qtd_contextos"))
    .toPandas()
)

pdf_prioridade["prioridade"] = (
    pdf_prioridade["prioridade"]
    .astype("category")
    .cat.set_categories(ordem_prioridade, ordered=True)
)

pdf_prioridade = pdf_prioridade.sort_values("prioridade")

fig, ax = plt.subplots(figsize=(9, 5))

ax.bar(pdf_prioridade["prioridade"].astype(str), pdf_prioridade["qtd_contextos"])

ax.set_title("Distribuição dos contextos por prioridade")
ax.set_xlabel("Prioridade")
ax.set_ylabel("Quantidade de contextos")

plt.tight_layout()

salvar_figura(fig, "distribuicao_prioridade")

plt.show()
plt.close(fig)

## 12.6 Aderência entre Risco Estimado e Resultado Observado

Para avaliar se o sinal produzido pelo modelo também é informativo em nível territorial, o risco médio estimado para cada município é comparado à taxa efetivamente observada de alunos não alfabetizados em 2024.

A linha diagonal representa correspondência perfeita entre risco estimado e resultado observado. Pontos próximos à linha indicam maior aderência entre o sinal do modelo e a situação observada.

Essa análise não representa uma avaliação de calibração formal, mas permite verificar se o modelo consegue capturar diferenças relativas de vulnerabilidade entre os municípios.

In [0]:
# Carrega novamente o produto persistido na Gold
df_risco_municipio_plot = spark.table("workspace.gold.ml_risco_municipio_2024")


# Constrói os indicadores de aderência
df_aderencia_municipal = (
    df_risco_municipio_plot.withColumn(
        "taxa_real_nao_alfabetizacao_pct",
        F.round(100 - F.col("taxa_real_alfabetizacao_pct"), 2),
    )
    .withColumn(
        "erro_risco_pp",
        F.round(
            F.abs(F.col("risco_medio_pct") - F.col("taxa_real_nao_alfabetizacao_pct")),
            2,
        ),
    )
    .withColumn(
        "erro_risco_pp",
        F.round(
            F.abs(F.col("risco_medio_pct") - F.col("taxa_real_nao_alfabetizacao_pct")),
            2,
        ),
    )
)

In [0]:

# Métricas agregadas de aderência
display(
    df_aderencia_municipal.agg(
        F.round(F.corr("risco_medio_pct", "taxa_real_nao_alfabetizacao_pct"), 3).alias(
            "correlacao_risco_observado"
        ),
        F.round(F.avg("erro_risco_pp"), 2).alias("mae_municipal_pp"),
    )
)


# Converte para Pandas para visualização
pdf_aderencia = (
    df_aderencia_municipal.select(
        "municipio",
        "sigla_uf",
        "qtd_alunos",
        "risco_medio_pct",
        "taxa_real_nao_alfabetizacao_pct",
    )
    .dropna()
    .toPandas()
)


# Gráfico
fig, ax = plt.subplots(figsize=(10, 7))

ax.scatter(
    pdf_aderencia["risco_medio_pct"],
    pdf_aderencia["taxa_real_nao_alfabetizacao_pct"],
    alpha=0.30,
    s=15,
)

limite = max(
    pdf_aderencia["risco_medio_pct"].max(),
    pdf_aderencia["taxa_real_nao_alfabetizacao_pct"].max(),
)

ax.plot([0, limite], [0, limite], linestyle="--")

ax.set_title("Risco estimado versus não alfabetização observada")

ax.set_xlabel("Risco médio estimado (%)")

ax.set_ylabel("Não alfabetização observada (%)")

plt.tight_layout()


# Salva PNG
caminho = images_dir / "aderencia_modelo_municipios.png"

fig.savefig(caminho, dpi=200, bbox_inches="tight")

print(f"PNG salvo: {caminho}")

plt.show()
plt.close(fig)

## 12.7 Aderência por faixas de risco

In [0]:
# Ordena os municípios pelo risco previsto
window_risco_decil = Window.orderBy(F.col("risco_medio_pct"))


# Divide os municípios em 10 grupos de risco
df_aderencia_decil = (
    df_aderencia_municipal.withColumn(
        "decil_risco", F.ntile(10).over(window_risco_decil)
    )
    .groupBy("decil_risco")
    .agg(
        F.count("*").alias("qtd_municipios"),
        F.round(F.avg("risco_medio_pct"), 2).alias("risco_medio_estimado_pct"),
        F.round(F.avg("taxa_real_nao_alfabetizacao_pct"), 2).alias(
            "nao_alfabetizacao_observada_pct"
        ),
    )
    .orderBy("decil_risco")
)


# Mostra a tabela usada no gráfico
display(df_aderencia_decil)


# Converte para Pandas
pdf_aderencia_decil = df_aderencia_decil.toPandas()


# Gráfico
fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(
    pdf_aderencia_decil["decil_risco"],
    pdf_aderencia_decil["risco_medio_estimado_pct"],
    marker="o",
    label="Risco estimado",
)

ax.plot(
    pdf_aderencia_decil["decil_risco"],
    pdf_aderencia_decil["nao_alfabetizacao_observada_pct"],
    marker="o",
    label="Não alfabetização observada",
)

ax.set_title("Risco estimado e não alfabetização observada por decil")

ax.set_xlabel("Decil de risco")

ax.set_ylabel("Percentual (%)")

ax.set_xticks(pdf_aderencia_decil["decil_risco"])

ax.legend()

plt.tight_layout()


# Salva PNG
caminho = images_dir / "aderencia_por_decil.png"

fig.savefig(caminho, dpi=200, bbox_inches="tight")

print(f"PNG salvo: {caminho}")

plt.show()
plt.close(fig)

# 13. Salvamento das Tabelas Analíticas

As principais tabelas consolidadas são exportadas para o repositório do projeto.

As tabelas utilizadas na comunicação dos resultados são mantidas em `reports/tables/`.

Essa organização facilita a reutilização dos artefatos no README, na documentação técnica e no vídeo executivo.

Feature Importance

In [0]:
(
    df_importancia.orderBy(F.desc("importancia"))
    .toPandas()
    .to_csv(tables_dir / "feature_importance.csv", index=False)
)

Risco por região

In [0]:
(
    df_risco_regiao.orderBy(F.desc("risco_medio_pct"))
    .toPandas()
    .to_csv(tables_dir / "risco_por_regiao.csv", index=False)
)

Top municípios por risco

In [0]:
(
    df_risco_municipio.orderBy(F.desc("risco_medio_pct"))
    .limit(50)
    .toPandas()
    .to_csv(tables_dir / "top_municipios_risco.csv", index=False)
)

Top exposição ao risco

In [0]:
(
    df_priorizacao.orderBy(F.desc("indice_exposicao_risco"))
    .limit(50)
    .toPandas()
    .to_csv(tables_dir / "top_exposicao_risco.csv", index=False)
)

Priorização completa

In [0]:
(
    df_priorizacao.orderBy(F.desc("ordem_prioridade"), F.desc("defasagem_meta_pp"))
    .toPandas()
    .to_csv(tables_dir / "priorizacao_educacional_2024.csv", index=False)
)

Aderência municipal

In [0]:
(
    df_aderencia_municipal
    .select(
        "municipio",
        "sigla_uf",
        "regiao",
        "qtd_alunos",
        "risco_medio_pct",
        "taxa_real_nao_alfabetizacao_pct",
        "erro_risco_pp"
    )
    .toPandas()
    .to_csv(
        tables_dir / "aderencia_municipal.csv",
        index=False
    )
)

# 14. Aplicação em Políticas Públicas

Os resultados desenvolvidos neste projeto permitem transformar as previsões do modelo em indicadores territoriais de apoio à tomada de decisão.

A análise mostrou que o risco educacional não está distribuído de maneira uniforme entre municípios e regiões. Além disso, os fatores mais relevantes para o modelo estão relacionados principalmente ao desempenho educacional histórico, ao contexto socioeconômico e ao rendimento escolar, complementados por características de infraestrutura educacional.

A solução permite duas formas complementares de priorização:

- **priorização por vulnerabilidade**, identificando contextos mais distantes das metas de alfabetização;
- **priorização por escala**, identificando territórios em que o risco educacional está associado a uma população estudantil maior.

Esses indicadores podem apoiar gestores na seleção de municípios para análises mais aprofundadas, acompanhamento pedagógico, direcionamento de programas educacionais e investigação das condições associadas aos resultados de alfabetização.

O modelo não deve ser utilizado como mecanismo automático de decisão ou distribuição de recursos. Sua função é atuar como instrumento adicional de inteligência analítica, orientando onde análises e intervenções mais detalhadas podem ser necessárias.

# 15. Limitações

A principal limitação do projeto está relacionada à granularidade das variáveis disponíveis.

Embora o target seja definido no nível do aluno, grande parte das features representa características do município e da rede de ensino. Dessa forma, alunos pertencentes ao mesmo contexto compartilham grande parte das informações utilizadas pelo modelo.

A incorporação de dados em nível de escola ou aluno foi investigada, porém identificadores mascarados em algumas fontes públicas impediram associações confiáveis entre diferentes bases.

O modelo final apresentou **ROC-AUC de 0,6221** no teste temporal de 2024, indicando capacidade discriminatória moderada. Por esse motivo, as probabilidades produzidas são utilizadas principalmente como scores relativos de risco, e não como estimativas perfeitamente calibradas das taxas municipais de alfabetização.

A análise por decis mostrou, entretanto, que contextos com maior risco estimado também apresentam, em média, maiores taxas observadas de não alfabetização, indicando utilidade do modelo para ordenação territorial de vulnerabilidade.

As importâncias das variáveis da Random Forest representam sua contribuição para as decisões do modelo e não devem ser interpretadas como relações causais.

A distância em relação às metas futuras também não representa uma previsão direta da taxa de alfabetização de anos posteriores. Ela funciona como um indicador de comparação entre o sinal atual do modelo e a meta estabelecida para o território.

# 16. Monitoramento e Evolução

Em uma aplicação produtiva, o modelo deverá ser monitorado periodicamente para identificar mudanças nos dados e possíveis perdas de desempenho ao longo do tempo.

O monitoramento pode incluir:

- evolução do ROC-AUC, recall e F1-score;
- distribuição das probabilidades previstas;
- comparação entre risco estimado e resultados posteriormente observados;
- alterações na distribuição das principais features;
- mudanças na importância relativa das variáveis;
- cobertura das fontes utilizadas no enriquecimento;
- acompanhamento das distâncias em relação às metas educacionais.

Novas edições da Avaliação da Alfabetização poderão ser utilizadas como novos testes temporais e, posteriormente, para retreinamento do modelo.

Como evolução, também podem ser exploradas novas fontes com maior granularidade, técnicas adicionais de calibração probabilística e métodos de interpretabilidade como SHAP Values.

# 17. Conclusão

O projeto desenvolveu uma pipeline completa de Ciência de Dados para predição de alfabetização, partindo da base analítica construída a partir da camada Gold e incorporando informações educacionais, socioeconômicas, demográficas e de infraestrutura.

A modelagem evoluiu a partir de uma primeira versão com baixo poder discriminatório. A análise do sinal das features levou à inclusão de indicadores históricos do IDEB, produzindo melhora consistente nos diferentes algoritmos avaliados.

Após comparação entre Regressão Logística, Decision Tree, Random Forest e Gradient-Boosted Trees, a Random Forest foi selecionada como modelo final. O processo incluiu otimização de hiperparâmetros, análise de overfitting e definição do threshold exclusivamente no conjunto de validação.

O modelo definitivo foi treinado com os dados de 2023 e avaliado em um conjunto temporal completamente isolado de 2024, atingindo **ROC-AUC de 0,6221** e mantendo estabilidade em relação à validação.

A interpretabilidade mostrou forte participação do desempenho histórico em Língua Portuguesa, do INSE e do rendimento escolar. Em conjunto, essas variáveis concentram a maior parcela da importância da Random Forest.

Mesmo com capacidade individual moderada, as análises territoriais mostraram que o score do modelo possui utilidade para diferenciar contextos de menor e maior vulnerabilidade. A análise por decis evidenciou aumento da não alfabetização observada à medida que cresce o risco estimado.

As previsões foram posteriormente transformadas em indicadores de risco municipal, análises regionais, comparação com metas educacionais e mecanismos de priorização por vulnerabilidade e escala.

Com isso, o projeto completa o ciclo entre **dados, modelagem, validação, interpretabilidade e aplicação estratégica**, transformando os resultados do modelo em inteligência analítica para apoiar a discussão e o planejamento de políticas públicas educacionais.